# **Day 68: Unit Testing with unittest**

A **test** is code that **checks** whether our other code gives the correct answer.

**Why write tests?** When we change code later, we can run the tests again in one second and know immediately if we **broke** something that was working before.

**Real-life example:** Before a train leaves, the staff check the brakes, lights and doors. Tests are a checklist for our code.

Python has a built-in testing tool called `unittest`.

## **The Functions We Want to Test**

In [1]:
import math

def ndvi(nir, red):
    if nir < 0 or red < 0:
        raise ValueError("Reflectance cannot be negative")
    if nir + red == 0:
        return 0.0
    return (nir - red) / (nir + red)

def haversine(lat1, lon1, lat2, lon2):
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * 6371 * math.asin(math.sqrt(a))

def is_wb_pin(pin):
    s = str(pin)
    return len(s) == 6 and s.isdigit() and s[:2] in {"70", "71", "72", "73", "74"}

## **Writing Tests**
1. Make a class that comes from `unittest.TestCase`.
2. Every method whose name starts with `test_` is one test.
3. Inside each test, use **assert methods** to check the results.

In [2]:
import unittest

class TestNDVI(unittest.TestCase):
    def test_dense_vegetation(self):
        self.assertAlmostEqual(ndvi(0.45, 0.05), 0.8)

    def test_water_is_negative(self):
        self.assertLess(ndvi(0.02, 0.06), 0)

    def test_range(self):
        for nir, red in [(0.5, 0.0), (0.0, 0.5), (0.3, 0.3)]:
            with self.subTest(nir=nir, red=red):
                self.assertTrue(-1 <= ndvi(nir, red) <= 1)

    def test_zero_pixel(self):
        self.assertEqual(ndvi(0, 0), 0.0)

    def test_negative_raises_error(self):
        with self.assertRaises(ValueError):
            ndvi(-0.1, 0.2)

class TestDistance(unittest.TestCase):
    def test_same_point_is_zero(self):
        self.assertEqual(haversine(22.57, 88.36, 22.57, 88.36), 0)

    def test_kolkata_siliguri(self):
        d = haversine(22.5726, 88.3639, 26.7271, 88.3953)
        self.assertTrue(455 < d < 470)       # About 462 km straight line

class TestPin(unittest.TestCase):
    def test_valid(self):
        for p in [700001, "721101", 734001]:
            with self.subTest(pin=p):
                self.assertTrue(is_wb_pin(p))

    def test_invalid(self):
        for p in [110001, "7000", "70000a", 7000011]:
            with self.subTest(pin=p):
                self.assertFalse(is_wb_pin(p))

# In a notebook, we need argv and exit=False
unittest.main(argv=["ignored"], verbosity=2, exit=False)

test_kolkata_siliguri (__main__.TestDistance.test_kolkata_siliguri) ... 

ok


test_same_point_is_zero (__main__.TestDistance.test_same_point_is_zero) ... 

ok


test_dense_vegetation (__main__.TestNDVI.test_dense_vegetation) ... 

ok


... (9 outputs left out)

ok


test_valid (__main__.TestPin.test_valid) ... 

ok


----------------------------------------------------------------------
Ran 9 tests in 0.006s

OK


All tests passed. Now let us see what happens when there is a **bug**.

In [3]:
def area_hectare(pixels, resolution_m):
    return pixels * resolution_m ** 2 / 1000        # Bug: should be 10,000

class TestArea(unittest.TestCase):
    def test_landsat(self):
        self.assertEqual(area_hectare(100, 30), 9)      # 100 x 900 sq m = 90,000 sq m = 9 ha

result = unittest.TextTestRunner(verbosity=2).run(unittest.TestLoader().loadTestsFromTestCase(TestArea))

test_landsat (__main__.TestArea.test_landsat) ... 

FAIL


FAIL: test_landsat (__main__.TestArea.test_landsat)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\Manas\AppData\Local\Temp\ipykernel_119348\2102423633.py", line 6, in test_landsat
    self.assertEqual(area_hectare(100, 30), 9)      # 100 x 900 sq m = 90,000 sq m = 9 ha
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: 90.0 != 9



----------------------------------------------------------------------
Ran 1 test in 0.001s

FAILED (failures=1)


The test **failed** and clearly showed `90.0 != 9`. This tells us immediately that the calculation is 10 times too big, so the bug must be in the division. Let us fix it and run the test again.

In [4]:
def area_hectare(pixels, resolution_m):
    return pixels * resolution_m ** 2 / 10_000        # Fixed

result = unittest.TextTestRunner(verbosity=2).run(unittest.TestLoader().loadTestsFromTestCase(TestArea))

test_landsat (__main__.TestArea.test_landsat) ... 

ok


----------------------------------------------------------------------
Ran 1 test in 0.001s

OK


## **setUp: Fresh Data for Every Test**
`setUp()` runs **before every test**, so each test starts with clean data.

In [5]:
class Cart:
    def __init__(self):
        self.items = {}
    def add(self, item, price, qty=1):
        old_qty = self.items.get(item, (price, 0))[1]
        self.items[item] = (price, old_qty + qty)
    def total(self):
        return sum(p * q for p, q in self.items.values())

class TestCart(unittest.TestCase):
    def setUp(self):
        self.cart = Cart()
        self.cart.add("Jhalmuri", 20)

    def test_add_twice(self):
        self.cart.add("Jhalmuri", 20)
        self.assertEqual(self.cart.items["Jhalmuri"], (20, 2))

    def test_total(self):
        self.cart.add("Cha", 10, qty=3)
        self.assertEqual(self.cart.total(), 50)

result = unittest.TextTestRunner(verbosity=2).run(unittest.TestLoader().loadTestsFromTestCase(TestCart))

test_add_twice (__main__.TestCart.test_add_twice) ... 

ok


test_total (__main__.TestCart.test_total) ... 

ok


----------------------------------------------------------------------
Ran 2 tests in 0.001s

OK


## **Common Assert Methods**
| Method | Checks that |
|---|---|
| `assertEqual(a, b)` | a == b |
| `assertTrue(x)` | x is True |
| `assertFalse(x)` | x is False |
| `assertIn(a, b)` | a is in b |
| `assertIsNone(x)` | x is None |
| `assertAlmostEqual(a, b)` | a and b are equal up to 7 decimal places (for floats) |
| `assertLess(a, b)` | a < b |
| `assertRaises(Error)` | the code raises that error |

In a real project, tests are kept in files like `test_something.py`, and we run all of them from the terminal with:
```
python -m unittest discover -v
```

In [6]:
# Quick question
# Why do we use assertAlmostEqual instead of assertEqual for 0.1 + 0.2 == 0.3?